# Bài 19/25 · LeetCode #108 — Convert Sorted Array to Binary Search Tree

**Mức độ:** Easy · **Chủ đề:** Binary Search Tree · Divide and Conquer · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/convert-sorted-array-to-binary-search-tree/](https://leetcode.com/problems/convert-sorted-array-to-binary-search-tree/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Cho `nums` là mảng số nguyên **tăng nghiêm ngặt**. Hãy tạo **cây tìm kiếm nhị phân (BST)** cân bằng chiều cao. Với mỗi node: các giá trị bên trái nhỏ hơn node, bên phải lớn hơn; độ cao hai nhánh tại mọi node chênh tối đa 1.

### Ví dụ quan trọng

`[-10,-3,0,5,9]` có thể trả cây gốc `0`, trái `-3` (con trái `-10`), phải `9` (con trái `5`). Đề chấp nhận nhiều cây cân bằng khác nhau.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [1]:
from __future__ import annotations
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

def tree(values):
    """Tạo cây từ danh sách level-order; None = ô trống."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); queue=deque([root]); pos=1
    while queue and pos < len(values):
        parent=queue.popleft()
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.left=TreeNode(v); queue.append(parent.left)
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.right=TreeNode(v); queue.append(parent.right)
    return root

def show_tree(root):
    """Đọc cấu trúc cây bằng BFS, không in quá nhiều None cuối."""
    if root is None:
        return []
    q=deque([root]); result=[]
    while q:
        p=q.popleft()
        if p is None:
            result.append(None)
        else:
            result.append(p.val); q.append(p.left); q.append(p.right)
    while result and result[-1] is None:
        result.pop()
    return result

### 2.1. `TreeNode` thật sự là gì? Học từ một nút đơn

**Binary Tree (cây nhị phân)** gồm các nút (*node*). Mỗi nút lưu đúng ba thông tin:
`val` (giá trị), `left` (tham chiếu đến con trái), `right` (tham chiếu đến con phải).

Quan trọng: `left`/`right` **không phải** là vị trí trong mảng; đó là liên kết đến một **đối tượng** `TreeNode` khác.
`None` nghĩa là *không có nút*. Một cây rỗng được biểu diễn bởi `root = None`.

Ví dụ cây `1` có con trái `2` và con phải `3`:

```text
      1
     / \
    2   3
```

Nhìn vào code dựng cây dưới đây, hãy dự đoán từng phép in trước khi chạy.

In [2]:
root_demo = TreeNode(1)
root_demo.left = TreeNode(2)
root_demo.right = TreeNode(3)
print("Gốc:", root_demo.val)
print("Con trái:", root_demo.left.val)
print("Con phải:", root_demo.right.val)
print("Con trái của nút 2:", root_demo.left.left)  # None

Gốc: 1
Con trái: 2
Con phải: 3
Con trái của nút 2: None


### 2.2. Tham chiếu (*reference*) và giá trị khác nhau thế nào?

`a = root_demo.left` **không sao chép** một nút mới: `a` và `root_demo.left` là hai tên cùng chỉ đến một đối tượng.
Vì vậy `a.val = 99` sẽ thay đổi giá trị trong cây gốc.

- `a is b`: hỏi **có cùng một đối tượng** hay không.
- `a.val == b.val`: hỏi **hai giá trị bằng nhau** hay không.
- Hai nút khác nhau có thể cùng giá trị; với so sánh cây, ta cần xét cả cấu trúc, không chỉ xét một con số.

Phần này đặc biệt quan trọng khi gắn nút con trong giải pháp chính.

In [3]:
x_demo = TreeNode(7)
y_demo = TreeNode(7)
z_demo = x_demo
print("Giá trị bằng nhau:", x_demo.val == y_demo.val)  # True
print("Là một đối tượng:", x_demo is y_demo)        # False
print("Chỉ đến cùng nút:", x_demo is z_demo)       # True
z_demo.val = 8
print("Giá trị của x sau khi đổi qua z:", x_demo.val)

Giá trị bằng nhau: True
Là một đối tượng: False
Chỉ đến cùng nút: True
Giá trị của x sau khi đổi qua z: 8


### 2.3. `None`, `is None` và thứ tự kiểm tra an toàn

Với `node = None`, viết `node.val` sẽ gây `AttributeError`: không thể truy cập `val` của vật không tồn tại.
Cách an toàn: kiểm tra `node is None` **trước**, rồi mới dùng `node.val`.

Trong Python, biểu thức `A and B` có **short-circuit** (đoản mạch): nếu `A` là `False` thì Python **không đánh giá** `B` nữa.
Tương tự, với `A or B`, nếu `A` là `True` thì không cần tính `B`.
Đây là lý do ta có thể xử lý trường hợp rỗng trước khi truy cập thuộc tính và tránh nhánh đệ quy vô ích.

In [4]:
nothing = None
print("Là nút rỗng?", nothing is None)
print("Không phải nút rỗng?", nothing is not None)
print("Đọc an toàn:", nothing.val if nothing is not None else "không có nút")
leaf_demo = TreeNode(42)
print("Lá có hai con rỗng:", leaf_demo.left is None and leaf_demo.right is None)

Là nút rỗng? True
Không phải nút rỗng? False
Đọc an toàn: không có nút
Lá có hai con rỗng: True


### 2.4. Hiểu danh sách level-order và vì sao phải giữ `None`

LeetCode thường hiển thị cây theo thứ tự từng tầng, trái trước phải sau (*level-order*).
Trong notebook, hàm `tree(values)` dựng cây từ dạng ấy để dễ thử, còn `show_tree(root)` đưa cây về danh sách để đọc.
**Đừng coi list đó là định nghĩa của cây**: thuật toán nộp LeetCode nhận `TreeNode`, không nhận danh sách.

Ví dụ `[1,2,None,3]` là: gốc 1 có con trái 2, nút 2 có con trái 3.
Nó khác `[1,None,2,3]`: gốc 1 có con phải 2, nút 2 có con trái 3.
Dù phần tử `3` xuất hiện cuối cả hai danh sách, vị trí con trỏ khác nhau hoàn toàn.

In [5]:
shape_a = tree([1, 2, None, 3])
shape_b = tree([1, None, 2, 3])
print("A:", show_tree(shape_a), "gốc trái:", shape_a.left)
print("B:", show_tree(shape_b), "gốc trái:", shape_b.left)
print("A.gốc.phải:", shape_a.right, "B.gốc.phải:", shape_b.right)
assert show_tree(shape_a) != show_tree(shape_b)

A: [1, 2, None, 3] gốc trái: TreeNode(2)
B: [1, None, 2, 3] gốc trái: None
A.gốc.phải: None B.gốc.phải: TreeNode(2)


### 2.5. Đệ quy (*recursion*) từ con số 0: hàm gọi lại chính nó

Đệ quy có hai mảnh **bắt buộc**:

1. **Base case** (trường hợp dừng): đơn giản đến mức trả lời ngay, không gọi tiếp.
2. **Recursive step** (bước rút gọn): gọi lại hàm trên bài toán nhỏ hơn, rồi dùng kết quả ấy.

**Call stack (ngăn xếp lời gọi)** giữ lại vị trí mà mỗi lần gọi đang chờ `return`.
Ví dụ `countdown(3)` cần đi `3 → 2 → 1 → 0` rồi quay trở ra.
Trên cây, bước nhỏ hơn tự nhiên là cây con `node.left` hoặc `node.right`.

Kí hiệu chiều cao `h`: một đường từ gốc xuống lá dài tối đa `h` nút, nên tầng gọi lồng nhau có thể đến `h`.
Đệ quy không tự biến mất bộ nhớ: mỗi lần gọi đang chờ chiếm một phần call stack.

In [6]:
def countdown_demo(n, level=0):
    print("  " * level + f"vào n={n}")
    if n == 0:
        print("  " * level + "đã đến base case")
        return 0
    result = 1 + countdown_demo(n - 1, level + 1)
    print("  " * level + f"trở lại n={n}, trả {result}")
    return result

assert countdown_demo(3) == 3

vào n=3
  vào n=2
    vào n=1
      vào n=0
      đã đến base case
    trở lại n=1, trả 1
  trở lại n=2, trả 2
trở lại n=3, trả 3


### 2.6. `Optional[TreeNode]` và `return` là gì?

`Optional[TreeNode]` (trong `typing`) mô tả kiểu dữ liệu có thể là một `TreeNode` **hoặc** `None`.
Đây là **type hint** (gợi ý kiểu), không làm Python tự kiểm tra cây hợp lệ trong lúc chạy.

`return` *trả kết quả về nơi gọi và kết thúc lượt chạy hiện tại*. `print` chỉ đưa thông tin lên màn hình;
một hàm chỉ `print(True)` mà không `return True` sẽ trả về `None` (không đúng kiểu `bool` mà nhiều bài đòi).

Vòng lặp `for`, phép `max`, phép `abs`, danh sách `[]`, biểu thức `x is None`, phép truy cập `.left` / `.right`
đều sẽ được giải thích lại ngay tại chỗ cần dùng ở phần sau.

In [7]:
def check_node_demo(node: Optional[TreeNode]) -> bool:
    if node is None:
        return False
    return node.val > 0

print("Kiểm tra None:", check_node_demo(None))
print("Kiểm tra nút 5:", check_node_demo(TreeNode(5)))
print("Kiểm tra nút -2:", check_node_demo(TreeNode(-2)))

Kiểm tra None: False
Kiểm tra nút 5: True
Kiểm tra nút -2: False


## 3. Python nền tảng — công cụ cần hiểu trước khi giải

`nums[lo:hi]` là slicing; `mid=(lo+hi)//2` chọn phần tử giữa. Đừng nhầm `mid` (index) với `nums[mid]` (giá trị).

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [8]:
nums=[-10,-3,0,5,9]
lo,hi=0,len(nums)-1;mid=(lo+hi)//2
print('mid index',mid,'mid value',nums[mid])
print('dãy bên trái:',nums[:mid],'bên phải:',nums[mid+1:])

mid index 2 mid value 0
dãy bên trái: [-10, -3] bên phải: [5, 9]


### 3.1. Phân biệt hai yêu cầu trong đề: **BST** và **height-balanced**

*Binary Search Tree (BST, cây tìm kiếm nhị phân)* có quy tắc cho **mọi nút**:
- Mọi giá trị trong cây con **trái nhỏ hơn** giá trị của nút.
- Mọi giá trị trong cây con **phải lớn hơn** giá trị của nút.
(Đầu vào LeetCode #108 là danh sách tăng dần **không trùng giá trị**.)

*Height-balanced* (cân bằng theo chiều cao) là: **tại MỌI nút**, độ cao hai cây con chênh **không quá 1**.
Không chỉ cần hai nhánh của gốc gần bằng nhau; các cây con phía dưới cũng phải cân bằng.

Ví dụ `[1,2,3,4,5]` có thể dựng cây BST cân bằng:
```text
       3
     /   \
    1     4
     \     \
      2     5
```
Trái gốc có 2 nút, phải gốc có 2 nút; từng cây con cũng có độ cao chênh không quá 1.

In [9]:
data_demo = [1,2,3,4,5]
balanced_demo = TreeNode(3,
    TreeNode(1, None, TreeNode(2)),
    TreeNode(4, None, TreeNode(5)))
print("Dãy đã sắp xếp:", data_demo)
print("Đầu ra minh họa:", show_tree(balanced_demo))
print("Gốc:", balanced_demo.val, "trái:", balanced_demo.left.val, "phải:", balanced_demo.right.val)

Dãy đã sắp xếp: [1, 2, 3, 4, 5]
Đầu ra minh họa: [3, 1, 4, None, 2, None, 5]
Gốc: 3 trái: 1 phải: 4


### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

Có thể chèn từng số theo thứ tự vào BST. Nhưng nếu đã tăng dần, cây dễ thành một chuỗi dài, **không cân bằng**.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [10]:
def baseline(nums):
    root=None
    for x in nums:
        if root is None:root=TreeNode(x);continue
        node=root
        while node.right is not None:node=node.right
        node.right=TreeNode(x)
    return root

In [11]:
bad=baseline([-10,-3,0,5,9]);print('BFS cây lệch:',show_tree(bad))

BFS cây lệch: [-10, None, -3, None, 0, None, 5, None, 9]


### 4.1. Vì sao chèn lần lượt 1, 2, 3, ... thành cây “hình dây”?

Trong baseline phía trên, mỗi phần tử mới lớn hơn các phần tử đã chèn, nên nó đi dọc nhánh **phải**
tới cuối rồi được gắn vào đó.

Ví dụ `nums=[1,2,3,4]`:
```text
   1
    \
     2
      \
       3
        \
         4
```

Đây vẫn là BST hợp lệ vì mọi con phải lớn hơn cha. Nhưng cây **không cân bằng**:
gốc có nhánh trái sâu 0, nhánh phải sâu 3, chênh lệch 3 (>1).

Ngoài ra, mỗi lần chèn phải tìm đến cuối nên tổng số bước truy cập có thể là `1+2+...+(n-1)`, tức `O(n²)`.
Ta nên sử dụng ngay tính chất đầu vào **đã sắp xếp** thay vì chèn từng phần tử như vậy.

In [12]:
long_chain = baseline([1,2,3,4])
print("Cây chèn tăng dần:", show_tree(long_chain))
print("Gốc có con trái?", long_chain.left is not None)
print("Cạnh phải: ", end='')
cur = long_chain
while cur is not None:
    print(cur.val, end=' -> ')
    cur = cur.right
print('None')

Cây chèn tăng dần: [1, None, 2, None, 3, None, 4]
Gốc có con trái? False
Cạnh phải: 1 -> 2 -> 3 -> 4 -> None


### 4.2. Bí quyết: chọn **phần tử giữa** để chia đều hai nửa

Một giá trị giữa dãy chia mảng thành **các số nhỏ hơn** ở bên trái và **các số lớn hơn** ở bên phải.
Như vậy đồng thời giải quyết được **hai mục tiêu**:
- Quy tắc BST: trái nhỏ, phải lớn.
- Quy tắc cân bằng: hai nhóm có kích thước bằng nhau hoặc lệch nhiều nhất 1.

Với `nums=[-10,-3,0,5,9]`, chỉ số từ `0` tới `4`, phần tử giữa có index `2` và giá trị `0`.
Bên trái `[-10,-3]`, bên phải `[5,9]`. Áp dụng đúng ý tưởng ấy **đệ quy** cho mỗi nửa.

Khi số phần tử chẵn, có **hai phần tử giữa**. Chọn giữa trái hoặc giữa phải **đều có thể hợp lệ**;
LeetCode chấp nhận **bất kỳ cây BST cân bằng** thỏa điều kiện, không bắt buộc đúng một hình.

In [13]:
def show_halves(nums):
    if not nums:
        return None
    left, right = 0, len(nums) - 1
    mid = (left + right) // 2
    print("nums:", nums)
    print("chỉ số mid:", mid, "giá trị ở mid:", nums[mid])
    print("nửa trái:", nums[:mid], "nửa phải:", nums[mid+1:])
    return mid

show_halves([-10,-3,0,5,9])
show_halves([1,2,3,4])

nums: [-10, -3, 0, 5, 9]
chỉ số mid: 2 giá trị ở mid: 0
nửa trái: [-10, -3] nửa phải: [5, 9]
nums: [1, 2, 3, 4]
chỉ số mid: 1 giá trị ở mid: 2
nửa trái: [1] nửa phải: [3, 4]


1

## 5. Vì sao cần chuyển sang kỹ thuật chính?

Chọn **phần tử giữa** làm gốc: nửa bên trái luôn nhỏ hơn, nửa bên phải luôn lớn hơn. Hai nửa chênh nhau tối đa 1 phần tử → cây cân bằng.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Dùng hai chỉ số `lo`,`hi` để tránh cắt mảng liên tục. Khi `lo>hi`, cây con rỗng. Cứ chọn giữa, tạo node, xây trái trên `[lo,mid-1]` và phải trên `[mid+1,hi]`.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

In [14]:
def solve(nums: list[int]) -> Optional[TreeNode]:
    def build(lo,hi):
        if lo>hi:return None
        mid=(lo+hi)//2
        node=TreeNode(nums[mid])
        node.left=build(lo,mid-1)
        node.right=build(mid+1,hi)
        return node
    return build(0,len(nums)-1)

### 6.1. Phân rã `build(lo,hi)` — vì sao không phải cắt list trong từng lần gọi?

Hàm phụ nhận **hai chỉ số** (`lo`, `hi`) đại diện đoạn con đang xử lý, gồm cả hai đầu.
Nó dùng **chung** mảng `nums`, không sao chép hai nửa bằng slicing mỗi lượt.

- `lo > hi`: đoạn không có phần tử ⇒ con trống ⇒ `None`.
- `mid = (lo+hi)//2`: lấy index giữa (phép chia nguyên; đã học ở bài #69).
- `TreeNode(nums[mid])`: tạo nút **mới** tại giá trị giữa.
- `node.left = build(lo,mid-1)`: dựng nhánh chứa giá trị nhỏ hơn.
- `node.right = build(mid+1,hi)`: dựng nhánh chứa giá trị lớn hơn.
- `return node`: trả gốc vừa tạo cho lời gọi cha.

**Vì sao tránh slicing?** `nums[:mid]` mỗi lần tạo list mới, tổng chi phí sao chép không cần thiết.
Dùng index giúp thời gian thuật toán vẫn `O(n)` với `n` là số phần tử đầu vào, và bộ nhớ phụ do đệ quy chỉ `O(log n)`.

In [15]:
def trace_one_interval(nums, lo, hi):
    if lo > hi:
        print(f"đoạn [{lo},{hi}] rỗng => None")
        return
    mid = (lo + hi) // 2
    print(f"lo={lo}, hi={hi}, mid={mid}; nums[mid]={nums[mid]}")
    print("đoạn trái:", (lo,mid-1), "đoạn phải:", (mid+1,hi))

sample_sorted = [-10,-3,0,5,9]
trace_one_interval(sample_sorted, 0, 4)
trace_one_interval(sample_sorted, 0, 1)
trace_one_interval(sample_sorted, 3, 4)

lo=0, hi=4, mid=2; nums[mid]=0
đoạn trái: (0, 1) đoạn phải: (3, 4)
lo=0, hi=1, mid=0; nums[mid]=-10
đoạn trái: (0, -1) đoạn phải: (1, 1)
lo=3, hi=4, mid=3; nums[mid]=5
đoạn trái: (3, 2) đoạn phải: (4, 4)


## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Phạm vi `lo..hi` | mid | Gốc mới | Hai lời gọi con |
|---|---:|---:|---|
| `0..4` | 2 | 0 | `[0..1]`, `[3..4]` |
| `0..1` | 0 | -10 | rỗng, `[1..1]` |
| `3..4` | 3 | 5 | rỗng, `[4..4]` |
| `lo>hi` | — | None | dừng |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

In [16]:
def trace_bst(nums):
    def build(lo,hi,depth=0):
        if lo>hi:
            print('  '*depth,f'[{lo},{hi}] -> None');return None
        mid=(lo+hi)//2
        print('  '*depth,f'[{lo},{hi}] -> mid={mid} giá trị={nums[mid]}')
        node=TreeNode(nums[mid]);node.left=build(lo,mid-1,depth+1);node.right=build(mid+1,hi,depth+1)
        return node
    return build(0,len(nums)-1)
print(show_tree(trace_bst([-10,-3,0,5,9])))

 [0,4] -> mid=2 giá trị=0
   [0,1] -> mid=0 giá trị=-10
     [0,-1] -> None
     [1,1] -> mid=1 giá trị=-3
       [1,0] -> None
       [2,1] -> None
   [3,4] -> mid=3 giá trị=5
     [3,2] -> None
     [4,4] -> mid=4 giá trị=9
       [4,3] -> None
       [5,4] -> None
[0, -10, 5, None, -3, None, 9]


### 7.1. Bảng trace bằng tay: những đoạn mảng và các nút sinh ra

`nums=[-10,-3,0,5,9]`. Mỗi hàng là một lần `build(lo,hi)` gặp đoạn **không rỗng**:

| Call | lo | hi | mid | nums[mid] | Gắn vào đâu? |
|---|---:|---:|---:|---:|---|
| `build(0,4)` | 0 | 4 | 2 | 0 | Gốc |
| `build(0,1)` | 0 | 1 | 0 | -10 | Con trái của 0 |
| `build(1,1)` | 1 | 1 | 1 | -3 | Con phải của -10 |
| `build(3,4)` | 3 | 4 | 3 | 5 | Con phải của 0 |
| `build(4,4)` | 4 | 4 | 4 | 9 | Con phải của 5 |

Những đoạn như `[0,-1]`, `[2,1]` trả `None` nên không tạo nút.

**Bất biến:** mọi lời gọi `build(lo,hi)` chỉ sử dụng các giá trị trong đúng đoạn `[lo,hi]`.
Không nút nào bị dùng 2 lần vì `mid` bị loại khỏi cả hai đoạn tiếp theo.

### 7.2. Một biến thể đúng: chọn giữa **bên phải** khi độ dài chẵn

Ví dụ `nums=[1,2,3,4]`:
- Giữa trái chọn index 1 (`2`) làm gốc.
- Giữa phải chọn index 2 (`3`) làm gốc.

Cả hai cách đều chia mảng thành hai nửa với độ dài chênh không quá 1.
Thuật toán đề bài **không yêu cầu cây duy nhất**. Khi test, phải kiểm chứng **BST** và **cân bằng**, không được bắt buộc
`show_tree(result)` giống hệt một mảng duy nhất lấy từ ví dụ.

In [17]:
def sorted_array_to_bst_right_mid(nums):
    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi + 1) // 2  # chọn giữa phải
        node = TreeNode(nums[mid])
        node.left = build(lo, mid - 1)
        node.right = build(mid + 1, hi)
        return node
    return build(0, len(nums) - 1)

vals = [1,2,3,4]
print("Cây chọn giữa trái:", show_tree(solve(vals)))
print("Cây chọn giữa phải:", show_tree(sorted_array_to_bst_right_mid(vals)))

Cây chọn giữa trái: [2, 1, 3, None, None, None, 4]
Cây chọn giữa phải: [3, 2, 4, 1]


### 7.3. Cách 3 — dựng cây *iterative* bằng stack các đoạn

Nếu không muốn dùng recursion, ta có thể lưu công việc đang chờ dưới dạng bộ
`(lo,hi,parent,side)` trong stack:
- `lo,hi` là đoạn mảng cần dựng thành một cây con.
- `parent` là nút cha cần gắn, `side` là `'left'` hoặc `'right'`.
- Mỗi công việc tạo nhiều nhất **một** `TreeNode` ở mid rồi thêm hai đoạn nhỏ hơn vào stack.

Đây vẫn là ý tưởng chia đôi — chỉ thay **call stack tự động** bằng **stack của chính mình**.
Cách đệ quy ở lời giải chính thường dễ đọc hơn cho người mới; cách iterative giúp hiểu chính xác
thông tin nào được truyền khi xuống cây con.

In [18]:
def sorted_array_to_bst_iterative(nums):
    if not nums:
        return None
    root = None
    jobs = [(0, len(nums)-1, None, "root")]
    while jobs:
        lo, hi, parent, side = jobs.pop()
        if lo > hi:
            continue
        mid = (lo + hi) // 2
        node = TreeNode(nums[mid])
        if parent is None:
            root = node
        elif side == "left":
            parent.left = node
        else:
            parent.right = node
        jobs.append((mid+1, hi, node, "right"))
        jobs.append((lo, mid-1, node, "left"))
    return root

for vals in [[1], [1,2,3,4], [-10,-3,0,5,9]]:
    print(vals, '->', show_tree(sorted_array_to_bst_iterative(vals)))

[1] -> [1]
[1, 2, 3, 4] -> [2, 1, 3, None, None, None, 4]
[-10, -3, 0, 5, 9] -> [0, -10, 5, None, -3, None, 9]


## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Theo tính tăng của `nums`, tất cả phần tử trước `mid` nhỏ hơn `nums[mid]`, tất cả phần tử sau lớn hơn. Theo quy nạp hai cây con là BST cân bằng. Chia hai đoạn dài khác nhau tối đa 1 giúp độ cao cây con khác nhau tối đa 1, nên cây tạo ra thỏa yêu cầu.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [19]:
class Solution:
    def sortedArrayToBST(self, nums: list[int]):
        def build(lo,hi):
            if lo>hi:return None
            mid=(lo+hi)//2
            node=TreeNode(nums[mid])
            node.left=build(lo,mid-1)
            node.right=build(mid+1,hi)
            return node
        return build(0,len(nums)-1)

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [20]:
def inorder(node):return inorder(node.left)+[node.val]+inorder(node.right) if node else []
def verify_balance(node):
    if node is None:return 0
    l=verify_balance(node.left);r=verify_balance(node.right)
    assert abs(l-r)<=1,'Cây không cân bằng'
    return 1+max(l,r)
for a in [[],[1],[1,3],[-10,-3,0,5,9],list(range(-15,16))]:
    for root in [solve(a),Solution().sortedArrayToBST(a)]:
        assert inorder(root)==a
        verify_balance(root)
print('PASS 5 ca BST, cả inorder và chiều cao')

PASS 5 ca BST, cả inorder và chiều cao


### 10.1. Đừng kiểm bằng hình cây duy nhất: kiểm **ba tính chất**

Một cây đầu ra hợp lệ phải đạt cả ba điều:
1. **Inorder** (trái → gốc → phải) thu lại **chính xác** `nums`.
2. Mọi nút có **độ cao trái/phải chênh không quá 1**.
3. Số nút sinh ra đúng `len(nums)` (mỗi số dùng một lần).

Với đầu vào là dãy tăng dần **không trùng giá trị**, điều (1) xác nhận đúng quy luật BST.
Ta thử cả ba bản: giữa trái, giữa phải, dùng stack. Đừng so sánh hai cây có cùng hình, vì
số phần tử chẵn có thể tạo ra nhiều BST hợp lệ.

In [21]:
def count_nodes_bst(node):
    if node is None:
        return 0
    return 1 + count_nodes_bst(node.left) + count_nodes_bst(node.right)

def check_constructed_bst(arr, root):
    assert inorder(root) == arr, (arr, 'sai inorder')
    assert count_nodes_bst(root) == len(arr), (arr, 'sai số nút')
    verify_balance(root)  # hàm có assert cân bằng ở mọi nút

for n in range(0, 45):
    arr = list(range(-n//2, -n//2 + n))
    for fn in [solve, sorted_array_to_bst_right_mid, sorted_array_to_bst_iterative]:
        check_constructed_bst(arr, fn(arr))
print("PASS 45 kích thước × 3 cách (BST, cân bằng, số nút)")

PASS 45 kích thước × 3 cách (BST, cân bằng, số nút)


### 10.2. Tại sao “mảng sorted nhưng chứa số trùng” là trường hợp khác?

Theo đề #108, `nums` chứa **các số nguyên phân biệt** được sắp theo thứ tự tăng dần.
Nếu cho mảng `[1,1,2]` rồi yêu cầu cây BST theo quy tắc **trái < nút < phải** thì không thể giữ đồng thời
hai số `1` mà vẫn thỏa điều kiện đó. Vì vậy không tự ý mở rộng ràng buộc để nói lời giải sai.

Hãy nhớ phân biệt **dữ liệu hợp lệ theo đề** và **bài toán mở rộng**.
Khi nhận một đề mới có giá trị trùng, ta phải đọc lại định nghĩa BST của đề trước khi chọn cách cài đặt.

In [22]:
valid = [-10, -3, 0, 5, 9]
print("Đầu vào hợp lệ:", valid)
print("Tăng nghiêm ngặt:", all(valid[i] < valid[i+1] for i in range(len(valid)-1)))
invalid_example = [1,1,2]
print("Đầu vào có trùng không theo giả định:", invalid_example)
print("Tăng nghiêm ngặt:", all(invalid_example[i] < invalid_example[i+1] for i in range(len(invalid_example)-1)))

Đầu vào hợp lệ: [-10, -3, 0, 5, 9]
Tăng nghiêm ngặt: True
Đầu vào có trùng không theo giả định: [1, 1, 2]
Tăng nghiêm ngặt: False


## 11. Phân tích Time/Space Complexity

Với `n` phần tử, mỗi giá trị tạo đúng một node: **Time O(n)**. **Space O(n)** cho các node đầu ra; bộ nhớ phụ recursion O(log n) vì luôn chia đôi. Cách chèn tăng dần có thể sâu O(n).

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ Trả danh sách thay vì `TreeNode`.
- ❌ `node=TreeNode(mid)`: mid là **chỉ số**, cần `nums[mid]`.
- ❌ Dùng `[0,mid]` cho nhánh trái: lặp lại giá trị gốc.
- ❌ So sánh BFS với **một** kết quả cứng: có nhiều đáp án hợp lệ.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Đoạn code sai: chọn phần tử đầu làm gốc rồi cứ nối bên phải

**Nhìn thấy đầu vào đã sắp xếp không có nghĩa nên chèn tuần tự**.
Cách sai dưới đây dựng BST hợp lệ nhưng vi phạm **yêu cầu cân bằng**.
Ví dụ `[1,2,3,4,5]` biến thành cây lệch phải, chênh chiều cao tại gốc rất lớn.

**Lỗi khác:** thiếu `return None` khi `lo>hi` sẽ đệ quy không dừng hoặc truy cập index sai.
**Lỗi off-by-one:** gọi `build(lo,mid)` thay vì `build(lo,mid-1)` sẽ dùng lại `mid` liên tục.

Sửa cả ba điểm: chọn `mid`, bỏ nó khỏi đoạn trái/phải, và có base case đoạn rỗng.

In [23]:
def difference_at_root(node):
    def h(n):
        return 0 if n is None else 1 + max(h(n.left), h(n.right))
    return 0 if node is None else abs(h(node.left)-h(node.right))

chain = baseline([1,2,3,4,5])
correct = solve([1,2,3,4,5])
print("Độ lệch chiều cao gốc — chèn tuần tự:", difference_at_root(chain))
print("Độ lệch chiều cao gốc — chọn mid:", difference_at_root(correct))
assert difference_at_root(chain) > 1
assert difference_at_root(correct) <= 1

Độ lệch chiều cao gốc — chèn tuần tự: 4
Độ lệch chiều cao gốc — chọn mid: 0


## 13. Nhận diện dạng bài và vận dụng

Đầu vào **đã sắp xếp**, đầu ra là cấu trúc **cân bằng** → chọn giữa, phân chia bài toán; dạng divide and conquer.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Tự tìm **inorder traversal** của cây vừa tạo: BST phải trả lại đúng `nums` đã sắp xếp. Kết quả shape có thể khác nhưng inorder không đổi.

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [24]:
root=solve([-10,-3,0,5,9]);print('Cây:',show_tree(root))
def inorder_demo(node):return inorder_demo(node.left)+[node.val]+inorder_demo(node.right) if node else []
print('Inorder:',inorder_demo(root))

Cây: [0, -10, 5, None, -3, None, 9]
Inorder: [-10, -3, 0, 5, 9]


### 14.1. Pattern nhận dạng: `sorted array → balanced BST` là chia để trị

*Divide and Conquer (chia để trị)*: chia bài toán lớn thành các bài con có cùng dạng,
giải các bài con và ghép lại.
Ở đây **dữ liệu đã sắp xếp** cho phép chọn ngay một gốc làm “mốc phân chia”.

| Vai trò | Bài #108 |
|---|---|
| Chia | Chọn mid của đoạn `[lo,hi]` |
| Giải | Dựng BST cân bằng cho nửa trái / nửa phải |
| Ghép | Dùng `TreeNode(nums[mid])`, nối hai gốc con |
| Dừng | `lo>hi → None` |

**Time:** `O(n)` vì mỗi phần tử biến thành một nút đúng một lần. **Extra space:** `O(log n)` do chiều sâu
lời gọi, không tính bộ nhớ `O(n)` của cây **đầu ra** (bắt buộc phải tạo).
Nếu một cách tính độ phức tạp tính cả output, hãy ghi rõ `O(n)` tổng bộ nhớ kể cả cây kết quả.

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

Cho `nums=[-10,-3,0,5,9]`, dự đoán `mid=(0+len(nums)-1)//2`, `nums[mid]`, `nums[:mid]` và `nums[mid+1:]`.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [25]:
nums=[-10,-3,0,5,9]
mid=(0+len(nums)-1)//2
print(mid,nums[mid])
print(nums[:mid],nums[mid+1:])

2 0
[-10, -3] [5, 9]


### Bài 2 — Trace bằng tay (hiểu bản chất)

Với `[-10,-3,0,5,9]`, viết gốc cây, gốc cây con trái, gốc cây con phải khi luôn chọn chỉ số giữa **làm tròn xuống**.

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Viết `exercise_sorted_array_to_bst(nums)` trả về **TreeNode** (không trả list) và dùng chỉ số `lo,hi` để hạn chế tạo lát cắt mảng.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [26]:
def exercise_sorted_array_to_bst(nums):
    # TODO: build(lo,hi); lo>hi -> None.
    # TODO: chọn mid, tạo TreeNode(nums[mid]), gắn left/right.
    raise NotImplementedError

### Bài 4 — TODO: Chuyển giao sang biến thể

Để kiểm tra BST, viết `exercise_inorder_values(root)` duyệt trái → gốc → phải; kết quả phải trùng với `nums` đã sắp xếp.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [27]:
def exercise_inorder_values(root):
    # TODO: trả về list các giá trị inorder, [] nếu root None.
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: `exercise_max_height_difference(root)` trả về **hiệu độ cao trái/phải lớn nhất** của bất kỳ node nào (cây rỗng là 0). Dùng một hàm phụ trả đồng thời độ cao và chênh lệch lớn nhất; bài này dẫn sang LeetCode #110.

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [28]:
def exercise_max_height_difference(root):
    # TODO: mỗi lần đệ quy trả về cặp (height, worst_difference).
    raise NotImplementedError

### Bài 6 — Dựng cây nhưng ưu tiên chọn **giữa phải**

**Yêu cầu:** viết `exercise_right_mid_bst(nums)` cho cây cân bằng bằng `mid=(lo+hi+1)//2`,
thay vì công thức giữa trái. Cả hai biến thể đều hợp lệ.

**Gợi ý 1:** giữ nguyên mọi quy tắc `build(lo,hi)` và base case `lo>hi`.

<details><summary>Gợi ý 2</summary>Dòng thay đổi duy nhất ở phép tính index <code>mid</code>.</details>

<details><summary>Gợi ý 3</summary>Gắn <code>left=build(lo,mid-1)</code>, <code>right=build(mid+1,hi)</code>, trả gốc.</details>

In [29]:
def exercise_right_mid_bst(nums):
    # TODO: dựng BST từ mảng sorted với mid giữa-phải.
    # Không thay đổi nums. Trả TreeNode hoặc None.
    raise NotImplementedError

# Sau khi làm: assert inorder(exercise_right_mid_bst([1,2,3,4])) == [1,2,3,4]

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

BST: mọi giá trị ở bên trái nhỏ hơn gốc; bên phải lớn hơn gốc. Cân bằng: chia kích thước gần một nửa.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

Với `lo,hi`, `mid=(lo+hi)//2`, lời gọi trái `build(lo,mid-1)` và phải `build(mid+1,hi)`.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

Inorder phải khôi phục mảng tăng dần. Chênh lệch độ cao tại mỗi node là `abs(height_left-height_right)`.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Dòng 1 `2 0`. Dòng 2 `[-10, -3] [5, 9]`. Chọn giữa để chia thành hai phần gần bằng nhau.

### Bài 2 — Trace tay

Gốc là `0`; dãy trái `[-10,-3]` lấy giữa làm tròn xuống là `-10`; dãy phải `[5,9]` lấy giữa làm tròn xuống là `5`. Hai lá tiếp theo là `-3` và `9`.

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [30]:
def answer_sorted_array_to_bst(nums):
    def build(lo,hi):
        if lo>hi: return None
        mid=(lo+hi)//2
        root=TreeNode(nums[mid])
        root.left=build(lo,mid-1)
        root.right=build(mid+1,hi)
        return root
    return build(0,len(nums)-1)

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [31]:
def answer_inorder_values(root):
    if root is None: return []
    return answer_inorder_values(root.left)+[root.val]+answer_inorder_values(root.right)

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [32]:
def answer_max_height_difference(root):
    def visit(node):
        if node is None: return 0,0
        lh,ld=visit(node.left)
        rh,rd=visit(node.right)
        return 1+max(lh,rh),max(ld,rd,abs(lh-rh))
    return visit(root)[1]

### Đáp án bài 6 — giữa phải, vẫn đúng

Hãy thử `n=2,3,4,5`: số phần tử của hai đoạn con chênh tối đa 1.
Ta không cần làm tròn về bên trái mới tạo được cây hợp lệ.

In [33]:
def answer_right_mid_bst(nums):
    def build(lo,hi):
        if lo>hi:
            return None
        mid=(lo+hi+1)//2
        node=TreeNode(nums[mid])
        node.left=build(lo,mid-1)
        node.right=build(mid+1,hi)
        return node
    return build(0,len(nums)-1)

for n in range(0,30):
    vals=list(range(n))
    check_constructed_bst(vals,answer_right_mid_bst(vals))
print("PASS bài 6: 30 kích thước mảng")

PASS bài 6: 30 kích thước mảng


### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [34]:
for arr in [[],[2],[1,3],[-10,-3,0,5,9],[-5,-2,1,7,20,33]]:
    root=answer_sorted_array_to_bst(arr)
    assert answer_inorder_values(root)==arr
    assert answer_max_height_difference(root)<=1
assert answer_max_height_difference(TreeNode(3,TreeNode(2,TreeNode(1))))==2
print("PASS: đáp án 3 bài coding của phần luyện tập")

PASS: đáp án 3 bài coding của phần luyện tập


### Tự đánh giá #108: chưa đạt nếu chỉ nhớ công thức `mid`

1. Vì sao chọn mốc giữa giúp cân bằng nhưng chọn mốc đầu mảng thì không?
2. Tại sao các phần tử nhỏ hơn mid đều thuộc cây con trái mà không cần kiểm tra từng số?
3. `lo>hi` biểu diễn trường hợp nào trong cây?
4. `build(lo,mid)` sai ở đâu? Hãy trace `nums=[1]` để thấy lặp mid.
5. Vì sao không thể kiểm thử bằng `show_tree(result)==một_mảng_cố_định`?
6. Phân biệt `O(log n)` **bộ nhớ phụ** và `O(n)` bộ nhớ của cây được tạo ra.

**Bài tự vẽ:** với `[1,2,3,4,5,6]`, tự dựng cây theo giữa trái và giữa phải; ghi ra inorder cho cả hai và kiểm tra độ lệch chiều cao của từng nút.

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** Chọn **phần tử giữa** làm gốc: nửa bên trái luôn nhỏ hơn, nửa bên phải luôn lớn hơn. Hai nửa chênh nhau tối đa 1 phần tử → cây cân bằng.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.